# 03 - Placement model (XGBoost + SHAP)
**Question:** holding the comp fixed, how much is a star upgrade, an item, or avoiding contestation worth in placement?

Needs `data/boards_clustered.csv`, which is written by the naming cell at the end of notebook 02.
Install once in the terminal: `python -m pip install xgboost shap scikit-learn`

Reading SHAP: placement is lower-is-better, so a **negative SHAP value helps** a board.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import shap
from sklearn.model_selection import GroupKFold, cross_val_predict
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

DATA = next(p for p in [Path("data"), Path("../data"), Path("D:/VS Code/data")] if (p / "boards_clustered.csv").exists())
boards = pd.read_csv(DATA / "boards_clustered.csv", index_col="board_id")
units = pd.read_csv(DATA / "units.csv")
units["unit"] = (units["unit"].str.replace(r"DA_(18_)?", "", regex=True)
                              .str.replace(r"18(?=[_:|]|$)", "", regex=True))
units = units[units["board_id"].isin(boards.index)]
print(boards.shape, units.shape)
boards[["cluster", "placement"]].head()

## 1. Contestation feature
For each board: how many *other* players in the same match played the same comp family.
Twin clusters (13 with 12, and 4 with 5) are counted as one family. Flex boards (cluster -1 and 10) get NaN,
because they are not a single comp, and XGBoost handles missing values natively.

In [ ]:
FLEX = [-1, 10]
family = boards["cluster"].replace({13: 12, 4: 5}).rename("family")
valid = ~boards["cluster"].isin(FLEX)

others = (boards[valid].groupby(["match_id", family[valid]])["placement"].transform("size") - 1)
boards["others_same_comp"] = np.nan
boards.loc[valid, "others_same_comp"] = others
boards["others_same_comp"].value_counts(dropna=False).sort_index()

## 2. Features
- `star_<unit>`: that unit's star level on the board (0 = absent). Rare units (under 2% of boards) are dropped.
- Item and star summaries per board.
- `level`, `n_units`, archetype, contestation.

Left out on purpose (they are outcomes, not causes): `last_round`, `gold_left`, `players_eliminated`, `total_damage_to_players`.

In [ ]:
tier = units.groupby(["board_id", "unit"])["tier"].max().unstack(fill_value=0)
tier = tier.loc[:, (tier > 0).mean() >= 0.02]
tier.columns = [f"star_{c}" for c in tier.columns]

summ = units.groupby("board_id").agg(
    total_items=("n_items", "sum"),
    n_full_items=("n_items", lambda t: (t >= 3).sum()),
    max_items_one_unit=("n_items", "max"),
    n_2star=("tier", lambda t: (t == 2).sum()),
    n_3star=("tier", lambda t: (t >= 3).sum()),
)

arch_col = "archetype" if "archetype" in boards.columns else "cluster"
arch = pd.get_dummies(boards[arch_col].astype(str), prefix="arch").astype(int)

X = pd.concat([boards[["level", "n_units", "others_same_comp"]], summ, tier, arch], axis=1).loc[boards.index]
fill0 = list(summ.columns) + list(tier.columns)
X[fill0] = X[fill0].fillna(0)
y = boards["placement"]
groups = boards["match_id"]
arch_cols = list(arch.columns)
print("Feature matrix:", X.shape)

## 3. Which model, and does the archetype add anything?
Cross-validated by match (all 8 boards of a game stay together), so scores are not inflated.

In [ ]:
cv = GroupKFold(n_splits=5)

def evaluate(model, data, name):
    pred = cross_val_predict(model, data, y, cv=cv, groups=groups)
    print(f"{name:30s} MAE = {mean_absolute_error(y, pred):.3f}   R2 = {r2_score(y, pred):.3f}")
    return pred

print(f"{'Predict the average':30s} MAE = {mean_absolute_error(y, np.full(len(y), y.mean())):.3f}   R2 = 0.000")

evaluate(make_pipeline(StandardScaler(), Ridge(alpha=10)), X.fillna(0), "Ridge (linear)")

xgbm = xgb.XGBRegressor(n_estimators=400, learning_rate=0.03, max_depth=4, subsample=0.8,
                        colsample_bytree=0.7, min_child_weight=10, random_state=42, n_jobs=-1)
evaluate(xgbm, X.drop(columns=arch_cols), "XGBoost, no archetype")
evaluate(xgbm, X[arch_cols], "Archetype only")
pred = evaluate(xgbm, X, "XGBoost, all features")

## 4. SHAP: what drives placement?
Fit on all data, then explain. Negative = better placement.

In [ ]:
xgbm.fit(X, y)
explainer = shap.TreeExplainer(xgbm)
sv = explainer(X)
shap.summary_plot(sv.values, X, max_display=20)

In [ ]:
imp = (pd.DataFrame({"feature": X.columns, "mean_abs_shap": np.abs(sv.values).mean(axis=0)})
         .sort_values("mean_abs_shap", ascending=False).reset_index(drop=True))
imp.head(20)

## 5. Dependence plots: the shape of each effect

In [ ]:
for f in ["n_3star", "total_items", "others_same_comp", "level"]:
    shap.dependence_plot(f, sv.values, X, interaction_index=None)

## 6. What is a star upgrade worth, per unit?
Average SHAP of each unit at 1, 2 and 3 stars (negative = better). `gain_1_to_2` is the placement change from upgrading.
Ignore rows with very few boards.

In [ ]:
cols = list(X.columns)
rows = []
for c in tier.columns:
    j = cols.index(c)
    for t in [1, 2, 3]:
        m = (X[c] == t).values
        if m.sum() >= 30:
            rows.append(dict(unit=c.replace("star_", ""), star=t, mean_shap=sv.values[m, j].mean()))
upg = pd.DataFrame(rows).pivot(index="unit", columns="star", values="mean_shap").round(3)
if 1 in upg.columns and 2 in upg.columns:
    upg["gain_1_to_2"] = (upg[2] - upg[1]).round(3)
    upg = upg.sort_values("gain_1_to_2")
upg.head(20)

In [ ]:
from sklearn.base import clone

cut = boards["last_round"].median()
late = (boards["last_round"] >= cut).values
Xl, yl, gl = X[late], y[late], groups[late]
print("Late-game boards:", late.sum(), "| baseline MAE:", round(float((yl - yl.mean()).abs().mean()), 3))

sets = {
    "all features": list(X.columns),
    "no level / n_units": [c for c in X.columns if c not in ("level", "n_units")],
    "archetype only": arch_cols,
}
for name, cols in sets.items():
    p = cross_val_predict(xgbm, Xl[cols], yl, cv=GroupKFold(5), groups=gl)
    print(f"{name:22s} MAE = {mean_absolute_error(yl, p):.3f}   R2 = {r2_score(yl, p):.3f}")

xl = clone(xgbm).fit(Xl, yl)
sv_l = shap.TreeExplainer(xl)(Xl)
shap.summary_plot(sv_l.values, Xl, max_display=15)

In [ ]:
imp[imp["feature"] == "others_same_comp"]

In [ ]:
rows = []
for feat in ["n_3star", "others_same_comp"]:
    j = list(Xl.columns).index(feat)
    t = (pd.DataFrame({feat: Xl[feat].values, "shap": sv_l.values[:, j]})
           .groupby(feat)["shap"].agg(["count", "mean"]).round(2).reset_index()
           .rename(columns={feat: "value"}))
    t.insert(0, "feature", feat)
    rows.append(t)
value_table = pd.concat(rows, ignore_index=True)
value_table.to_csv(DATA / "value_table.csv", index=False)
value_table

In [ ]:
reroll = boards["cluster"].isin([0, 3, 2, 6])
chk = pd.DataFrame({"n_3star": X["n_3star"], "placement": y, "reroll": reroll, "late": late})
chk = chk[chk["late"]]
chk["n3"] = chk["n_3star"].clip(upper=3)
print(chk.groupby(["reroll", "n3"])["placement"].agg(["count", "mean"]).round(2))

In [ ]:
u3 = units[units["tier"] >= 3].copy()
u3["reroll"] = u3["board_id"].map(boards["cluster"].isin([0, 3, 2, 6]))
u3["placement"] = u3["board_id"].map(boards["placement"])
print(pd.crosstab(u3["rarity"], u3["reroll"]))
print()
print(u3.groupby("rarity")["placement"].agg(["count", "mean"]).round(2))

In [ ]:
from sklearn.base import clone
rng = np.random.default_rng(0)
matches = groups.unique()
rows = []
for frac in [0.25, 0.5, 0.75, 1.0]:
    maes = []
    for rep in range(3):
        keep = rng.choice(matches, size=int(len(matches) * frac), replace=False)
        m = groups.isin(keep).values
        pred = cross_val_predict(clone(xgbm), X[m], y[m], cv=GroupKFold(5), groups=groups[m])
        maes.append(mean_absolute_error(y[m], pred))
    rows.append(dict(matches=int(len(matches) * frac), mae=round(float(np.mean(maes)), 3)))
pd.DataFrame(rows)

## Reading the results
- Does XGBoost beat Ridge? (Your plan's cited research says it should.)
- Does "XGBoost, all features" beat "no archetype"? If not, comp choice adds little beyond board quality.
- Top of the SHAP list: is it star levels, items, level, or contestation?
- Does `others_same_comp` show a cost per competitor that matches the roughly 0.25-0.3 you measured?

Next: notebook 04, the Markov chain rolling model, which feeds the contestation cost into the shop probabilities.